# Week 5 — Day 2: Filtering, Sorting & Grouping
### Code to AI — Python & Data Science (from Zero to Portfolio)
**Instructor:** Ali Hasnain — Lead AI Engineer & CTO

---

## What you will learn today
1. `.loc` and `.iloc` — selecting rows
2. **Boolean filtering** — the single most important Pandas skill
3. Combining conditions with `&`, `|`, `~`
4. `.isin()` and `.between()`
5. Sorting with `sort_values()` and `.nlargest()`
6. Adding, renaming and dropping columns
7. **`groupby`** — split, apply, combine
8. A first peek at missing values

**Session plan (75 minutes)**

| Time | Section |
|------|---------|
| 0:00 – 0:05 | Recap of Day 1 |
| 0:05 – 0:15 | `.loc` vs `.iloc` |
| 0:15 – 0:35 | Boolean filtering (the big one) |
| 0:35 – 0:45 | Sorting |
| 0:45 – 0:52 | Adding / renaming / dropping columns |
| 0:52 – 0:68 | `groupby` |
| 0:68 – 0:75 | Missing values peek + wrap-up |

---

Yesterday we learned to **look** at data. Today we learn to **ask questions** of it.

In [46]:
import pandas as pd # data manuplation
import numpy as np # numrical python
import seaborn as sns # visualization --- we are using it for data source, titanic

titanic = sns.load_dataset("titanic")
print("Loaded:", titanic.shape)
titanic.head(3)

Loaded: (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True


## Recap in 3 lines

```python
df["age"]           # one column -> Series
df[["age","fare"]]  # many columns -> DataFrame
df.head()  df.shape  df.info()  df.describe()  df.isna().sum()
```

Notice what is missing: we can select **columns**, but not **rows**. That is today.

## 1. Selecting rows: `.loc` and `.iloc`

Two tools, one letter apart, and beginners mix them up constantly. Learn the difference now:

| Tool | Selects by | Think |
|------|-----------|-------|
| `.loc[]` | **label** (the index name) | **l** = label |
| `.iloc[]` | **integer position** | **i** = integer |

For the Titanic, the index is just `0, 1, 2, ...`, so they look similar.
The difference will bite you later — so understand it now.

In [49]:
type(titanic)

pandas.DataFrame

In [48]:
type(titanic['age'])

pandas.Series

In [51]:
titanic.head(1)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.25,S,Third,man,True,NaN,Southampton,no,False


In [50]:
# .iloc — by position
print("Row at position 0:")
print(titanic.iloc[0])

Row at position 0:
survived                 0
pclass                   3
sex                   male
age                   22.0
sibsp                    1
parch                    0
fare                  7.25
embarked                 S
class                Third
who                    man
adult_male            True
deck                   NaN
embark_town    Southampton
alive                   no
alone                False
Name: 0, dtype: object


In [52]:
# .iloc with a slice — first 3 rows. Stop is EXCLUSIVE, like a list.
titanic.iloc[0:3]

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True


In [54]:
# .loc — by label. Stop is INCLUSIVE. This is the big difference!
titanic.loc[0:5]

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
5,0,3,male,NaN,0,0,8.4583,Q,Third,man,True,NaN,Queenstown,no,True


Look carefully: `.iloc[0:3]` gave **3 rows**. `.loc[0:3]` gave **4 rows**.

- `.iloc` slicing excludes the stop (like a Python list)
- `.loc` slicing includes the stop (because it is a label, not a position)

This is the number one Pandas mistake. Write it down.

In [55]:
# Rows AND columns together: [rows, columns]
titanic.loc[0:4, ["sex", "age", "survived"]]

,sex,age,survived
0,male,22.0,0
1,female,38.0,1
2,female,26.0,1
3,female,35.0,1
4,male,35.0,0


In [56]:
# Same idea with .iloc, but using positions for both
# Rows 0-2, columns 0-3
titanic.iloc[0:3, 0:4]

,survived,pclass,sex,age
0,0,3,male,22.0
1,1,1,female,38.0
2,1,3,female,26.0


In [57]:
# A single cell
print("Age of passenger at position 4:", titanic.iloc[4]["age"])
print("Same thing with .loc:", titanic.loc[4, "age"])

Age of passenger at position 4: 35.0
Same thing with .loc: 35.0


## 2. Boolean filtering — the core skill

This is the most important thing in Week 5. Everything else is decoration.

**The idea, in two steps:**
1. Write a condition on a column. Pandas gives you back a Series of `True`/`False` — one per row.
2. Put that True/False Series inside `df[...]`. Pandas keeps only the `True` rows.

Let's see step 1 on its own first.

In [59]:
# Step 1: the condition alone. This is called a MASK.
mask = titanic["age"] > 30
print(mask.head(20))
print()
print("Type:", type(mask))
print("How many True?", mask.sum())

0     False
1      True
2     False
3      True
4      True
5     False
6      True
7     False
8     False
9     False
10    False
11     True
12    False
13     True
14    False
15     True
16    False
17    False
18     True
19    False
Name: age, dtype: bool

Type: <class 'pandas.Series'>
How many True? 305


Read that output. Row 0 is `False` (that passenger is 22, not over 30).
Row 1 is `True` (that passenger is 38).

`mask.sum()` counts the `True` values, because `True` acts like 1. Another useful trick.

Now step 2 — feed the mask back into the DataFrame.

In [60]:
# Step 2: use the mask to filter
over_30 = titanic[mask]
print("Original rows:", titanic.shape[0])
print("Filtered rows:", over_30.shape[0])
over_30.head()

Original rows: 891
Filtered rows: 305


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
6,0,1,male,54.0,0,0,51.8625,S,First,man,True,E,Southampton,no,True
11,1,1,female,58.0,0,0,26.5500,S,First,woman,False,C,Southampton,yes,True


In [62]:
titanic[titanic['age']> 30].tail(10)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
862,1,1,female,48.0,0,0,25.9292,S,First,woman,False,D,Southampton,yes,True
865,1,2,female,42.0,0,0,13.0000,S,Second,woman,False,NaN,Southampton,yes,True
867,0,1,male,31.0,0,0,50.4958,S,First,man,True,A,Southampton,no,True
871,1,1,female,47.0,1,1,52.5542,S,First,woman,False,D,Southampton,yes,False
872,0,1,male,33.0,0,0,5.0000,S,First,man,True,B,Southampton,no,True
873,0,3,male,47.0,0,0,9.0000,S,Third,man,True,NaN,Southampton,no,True
879,1,1,female,56.0,0,1,83.1583,C,First,woman,False,C,Cherbourg,yes,False
881,0,3,male,33.0,0,0,7.8958,S,Third,man,True,NaN,Southampton,no,True
885,0,3,female,39.0,0,5,29.1250,Q,Third,woman,False,NaN,Queenstown,no,False
890,0,3,male,32.0,0,0,7.7500,Q,Third,man,True,NaN,Queenstown,no,True


In [63]:
# In real code we write it in one line. Same thing.
titanic[titanic["age"] > 30].head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
6,0,1,male,54.0,0,0,51.8625,S,First,man,True,E,Southampton,no,True
11,1,1,female,58.0,0,0,26.5500,S,First,woman,False,C,Southampton,yes,True


### Read this out loud

```python
titanic[titanic["age"] > 30]
```
> "From titanic, give me the rows where titanic's age is greater than 30."

The outer `titanic[...]` is the **selecting**. The inner part is the **condition**.
Say it out loud every time until it feels natural.

### More conditions

In [64]:
# Equality uses == (double equals)
females = titanic[titanic["sex"] == "female"]
print("Female passengers:", females.shape[0])

# Not equal
not_third = titanic[titanic["class"] != "Third"]
print("Not in Third class:", not_third.shape[0])

# Survivors
survivors = titanic[titanic["survived"] == 1]
print("Survivors:", survivors.shape[0])

Female passengers: 314
Not in Third class: 400
Survivors: 342


## 3. Combining conditions: `&`, `|`, `~`

To combine conditions, Pandas uses symbols instead of words:

| Python word | Pandas symbol | Meaning |
|-------------|---------------|---------|
| `and` | `&` | both must be true |
| `or` | `\|` | at least one must be true |
| `not` | `~` | flip it |

**Two rules you must not forget:**
1. Use `&` `|` `~` — **not** the words `and` `or` `not`. The words will throw an error.
2. Wrap **every condition in its own brackets**. `()` around each one. Always.

In [75]:
female_servived = titanic[(titanic['sex']== "female") & (titanic['survived']== 1)]
female_servived

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
8,1,3,female,27.0,0,2,11.1333,S,Third,woman,False,NaN,Southampton,yes,False
9,1,2,female,14.0,1,0,30.0708,C,Second,child,False,NaN,Cherbourg,yes,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
874,1,2,female,28.0,1,0,24.0000,C,Second,woman,False,NaN,Cherbourg,yes,False
875,1,3,female,15.0,0,0,7.2250,C,Third,child,False,NaN,Cherbourg,yes,True
879,1,1,female,56.0,0,1,83.1583,C,First,woman,False,C,Cherbourg,yes,False
880,1,2,female,25.0,0,1,26.0000,S,Second,woman,False,NaN,Southampton,yes,False


In [76]:
male_servived = titanic[(titanic['sex']== "male") & (titanic['survived']== 1)]
male_servived

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
17,1,2,male,NaN,0,0,13.0000,S,Second,man,True,NaN,Southampton,yes,True
21,1,2,male,34.0,0,0,13.0000,S,Second,man,True,D,Southampton,yes,True
23,1,1,male,28.0,0,0,35.5000,S,First,man,True,A,Southampton,yes,True
36,1,3,male,NaN,0,0,7.2292,C,Third,man,True,NaN,Cherbourg,yes,True
55,1,1,male,NaN,0,0,35.5000,S,First,man,True,C,Southampton,yes,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
838,1,3,male,32.0,0,0,56.4958,S,Third,man,True,NaN,Southampton,yes,True
839,1,1,male,NaN,0,0,29.7000,C,First,man,True,C,Cherbourg,yes,True
857,1,1,male,51.0,0,0,26.5500,S,First,man,True,E,Southampton,yes,True
869,1,3,male,4.0,1,1,11.1333,S,Third,child,False,NaN,Southampton,yes,False


In [79]:
# AND — female AND survived
female_survivors = titanic[(titanic["sex"] == "female") & (titanic["survived"] == 1)]
print("Female survivors:", female_survivors.shape[0])
female_survivors[["sex", "age", "class", "survived"]].head()

Female survivors: 233


,sex,age,class,survived
1,female,38.0,First,1
2,female,26.0,Third,1
3,female,35.0,First,1
8,female,27.0,Third,1
9,female,14.0,Second,1


Count the brackets:
```python
titanic[ (titanic["sex"] == "female") & (titanic["survived"] == 1) ]
        ^--- condition 1 ---------^   ^--- condition 2 ----------^
```
Each condition gets its own `()`. Then `&` joins them. Then the whole thing goes inside `titanic[...]`.

If you forget the inner brackets you get a confusing error. It happens to everyone.

In [80]:
# OR — First class OR Second class
upper = titanic[(titanic["class"] == "First") | (titanic["class"] == "Second")]
print("First or Second class:", upper.shape[0])
upper[["survived", "age", "class"]].head()

First or Second class: 400


,survived,age,class
1,1,38.0,First
3,1,35.0,First
6,0,54.0,First
9,1,14.0,Second
11,1,58.0,First


In [81]:
# NOT — everyone who did NOT survive
died = titanic[~(titanic["survived"] == 1)]
print("Did not survive:", died.shape[0])

Did not survive: 549


In [82]:
# Three conditions at once
young_female_first = titanic[
    (titanic["sex"] == "female") &
    (titanic["age"] < 30) &
    (titanic["class"] == "First")
]
print("Young women in First class:", young_female_first.shape[0])
young_female_first[["sex", "age", "class", "survived"]].head()

Young women in First class: 30


,sex,age,class,survived
88,female,23.0,First,1
136,female,19.0,First,1
151,female,22.0,First,1
290,female,26.0,First,1
291,female,19.0,First,1


### `.isin()` — a shortcut for many ORs

Writing `(x == "a") | (x == "b") | (x == "c")` gets ugly fast. `.isin()` does it cleanly.

In [85]:
# The long way
long_way = titanic[(titanic["class"] == "First") | (titanic["class"] == "Second")]

# The clean way
clean_way = titanic[titanic["class"].isin(["First", "Second"])]

print("Long way :", long_way.shape[0])
print("Clean way:", clean_way.shape[0])

Long way : 400
Clean way: 400


### `.between()` — a shortcut for ranges

In [86]:
# The long way
teens_long = titanic[(titanic["age"] >= 13) & (titanic["age"] <= 19)]

# The clean way (between is INCLUSIVE on both ends)
teens = titanic[titanic["age"].between(13, 19)]
print("Teenagers:", teens_long.shape[0])
print("Teenagers:", teens.shape[0])

Teenagers: 95
Teenagers: 95


### Combine filtering with the summaries from Day 1

This is where it gets powerful. Filter, then summarize.

In [87]:
# Survival rate of women vs men
women = titanic[titanic["sex"] == "female"]
men = titanic[titanic["sex"] == "male"]

print("Women survival rate:", round(women["survived"].mean(), 3))
print("Men survival rate  :", round(men["survived"].mean(), 3))

Women survival rate: 0.742
Men survival rate  : 0.189


That is a real finding. 74% of women survived, 19% of men.
Two lines of code told us something historically true about that night.

**This is data science.** Filter, summarize, compare, conclude.

In [88]:
# Average fare in each class
for c in ["First", "Second", "Third"]:
    subset = titanic[titanic["class"] == c]
    print(c, "class average fare:", round(subset["fare"].mean(), 2))

First class average fare: 84.15
Second class average fare: 20.66
Third class average fare: 13.68


## 4. Sorting

`sort_values()` orders rows by a column.

In [90]:
# Oldest passengers first
titanic.sort_values(by="age", ascending=False)[["age", "sex", "class", "survived"]].head()

,age,sex,class,survived
630,80.0,male,First,1
851,74.0,male,Third,0
493,71.0,male,First,0
96,71.0,male,First,0
116,70.5,male,Third,0


In [91]:
# Oldest passengers first
titanic.sort_values(by="age")[["age", "sex", "class", "survived"]].head()

,age,sex,class,survived
803,0.42,male,Third,1
755,0.67,male,Second,1
644,0.75,female,Third,1
469,0.75,female,Third,1
78,0.83,male,Second,1


In [97]:
# Cheapest fares first (ascending=True is the default, so we can leave it out)
titanic.sort_values(by="fare")[["fare", "class", "sex"]].head(25)

,fare,class,sex
815,0.0000,First,male
806,0.0000,First,male
413,0.0000,Second,male
481,0.0000,Second,male
302,0.0000,Third,male
271,0.0000,Third,male
277,0.0000,Second,male
822,0.0000,First,male
263,0.0000,First,male
466,0.0000,Second,male


### Sorting by two columns

Pass a **list**. It sorts by the first, then breaks ties with the second.

In [98]:
# Sort by class, then within each class by fare (highest first)
titanic.sort_values(by=["class", "fare"], ascending=[True, False])[["class", "fare", "sex", "age"]].head(10)

,class,fare,sex,age
258,First,512.3292,female,35.0
679,First,512.3292,male,36.0
737,First,512.3292,male,35.0
27,First,263.0000,male,19.0
88,First,263.0000,female,23.0
341,First,263.0000,female,24.0
438,First,263.0000,male,64.0
311,First,262.3750,female,18.0
742,First,262.3750,female,21.0
118,First,247.5208,male,24.0


### `.nlargest()` and `.nsmallest()` — a shortcut

When you only want the top N, this is shorter and faster than sorting the whole table.

In [100]:
# The 5 most expensive tickets
titanic.nlargest(10, "fare")[["fare", "class", "sex", "survived"]]

,fare,class,sex,survived
258,512.3292,First,female,1
679,512.3292,First,male,1
737,512.3292,First,male,1
27,263.0000,First,male,0
88,263.0000,First,female,1
341,263.0000,First,female,1
438,263.0000,First,male,0
311,262.3750,First,female,1
742,262.3750,First,female,1
118,247.5208,First,male,0


In [101]:
# The 5 youngest passengers
titanic.nsmallest(5, "age")[["age", "who", "class", "survived"]]

,age,who,class,survived
803,0.42,child,Third,1
755,0.67,child,Second,1
469,0.75,child,Third,1
644,0.75,child,Third,1
78,0.83,child,Second,1


## 5. Adding, renaming and dropping columns

### Adding a column

Just assign to a new name. The calculation happens for every row at once.

In [102]:
# Work on a copy so we don't damage the original
df = titanic.copy()
df

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [103]:
df.columns

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alive', 'alone'],
      dtype='str')

In [106]:


# A new column from a calculation
df["family_size"] = df["sibsp"] + df["parch"] + 1   # +1 for the passenger themselves

df[["sibsp", "parch", "family_size"]].sample(5)

,sibsp,parch,family_size
865,0,0,1
100,0,0,1
624,0,0,1
753,0,0,1
10,1,1,3


In [107]:
df.columns

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alive', 'alone', 'family_size'],
      dtype='str')

In [109]:
titanic.columns

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alive', 'alone'],
      dtype='str')

In [108]:
df

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,family_size
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,2
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,2
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,1
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,2
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True,1
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True,1
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False,4
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True,1


In [110]:
# Another one: fare per family member
df["fare_per_person"] = df["fare"] / df["family_size"]

df[["fare", "family_size", "fare_per_person"]].head()

,fare,family_size,fare_per_person
0,7.2500,2,3.62500
1,71.2833,2,35.64165
2,7.9250,1,7.92500
3,53.1000,2,26.55000
4,8.0500,1,8.05000


In [111]:
df.columns

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alive', 'alone', 'family_size', 'fare_per_person'],
      dtype='str')

### Why `.copy()`?

If you write `df = titanic` you do **not** get a new table — you get a second name
pointing at the **same** table. Change one, and the other changes too.

`.copy()` makes a real, separate copy. Use it whenever you plan to modify a DataFrame.

### Renaming columns

In [112]:
# Rename takes a dictionary: {old_name: new_name}
df = df.rename(columns={"sibsp": "siblings_spouses", "parch": "parents_children"})
print(df.columns.tolist())

['survived', 'pclass', 'sex', 'age', 'siblings_spouses', 'parents_children', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone', 'family_size', 'fare_per_person']


### Dropping columns

In [113]:
# Drop columns you don't need. Pass a list.
df_small = df.drop(columns=["deck", "embark_town", "alive", "adult_male"])
print("Before:", df.shape)
print("After :", df_small.shape)
df_small.head(3)

Before: (891, 17)
After : (891, 13)


,survived,pclass,sex,age,siblings_spouses,parents_children,fare,embarked,class,who,alone,family_size,fare_per_person
0,0,3,male,22.0,1,0,7.2500,S,Third,man,False,2,3.62500
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,2,35.64165
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,True,1,7.92500


## 6. `groupby` — split, apply, combine

Look back at what we did earlier:

```python
women = titanic[titanic["sex"] == "female"]
men   = titanic[titanic["sex"] == "male"]
print(women["survived"].mean())
print(men["survived"].mean())
```

That worked — but we had to know the groups in advance and write a line for each.
What if there were 50 groups?

**`groupby` does it automatically.** The idea has three steps:

1. **Split** — break the table into groups (one per unique value)
2. **Apply** — run a calculation on each group
3. **Combine** — stack the answers back into one result

```
        SPLIT              APPLY           COMBINE
                       ┌─ female ─┐    ┌─ mean ─┐
   titanic  ────────►  │          │──► │        │──►  female  0.74
                       └─ male ───┘    └─ mean ─┘     male    0.19
```

In [114]:
# The whole thing in one line
titanic.groupby("sex")["survived"].mean()

sex
female    0.742038
male      0.188908
Name: survived, dtype: float64

Read it left to right:
```python
titanic.groupby("sex")["survived"].mean()
        ^-- split by sex
                       ^-- look at the survived column
                                    ^-- take the mean of each group
```

One line replaced six. And it found the groups by itself.

In [115]:
# Survival by class
titanic.groupby("class")["survived"].mean()

class
First     0.629630
Second    0.472826
Third     0.242363
Name: survived, dtype: float64

In [116]:
# Average fare by class
titanic.groupby("class")["fare"].mean()

class
First     84.154687
Second    20.662183
Third     13.675550
Name: fare, dtype: float64

In [117]:
# Average age by who (man/woman/child)
titanic.groupby("who")["age"].mean()

who
child     6.369518
man      33.173123
woman    32.000000
Name: age, dtype: float64

### Any summary works, not just `.mean()`

In [118]:
print("Count per class:")
print(titanic.groupby("class")["survived"].count())
print()
print("Max fare per class:")
print(titanic.groupby("class")["fare"].max())
print()
print("Total survivors per class:")
print(titanic.groupby("class")["survived"].sum())

Count per class:
class
First     216
Second    184
Third     491
Name: survived, dtype: int64

Max fare per class:
class
First     512.3292
Second     73.5000
Third      69.5500
Name: fare, dtype: float64

Total survivors per class:
class
First     136
Second     87
Third     119
Name: survived, dtype: int64


### Grouping by two columns

Pass a list. You get a group for every **combination**.

In [119]:
# Survival by sex AND class
titanic.groupby(["sex", "class"])["survived"].mean()

sex     class 
female  First     0.968085
        Second    0.921053
        Third     0.500000
male    First     0.368852
        Second    0.157407
        Third     0.135447
Name: survived, dtype: float64

Read this table slowly. It is one of the most famous results in data science.

- A woman in First class: about 97% survived
- A man in Third class: about 14% survived

Same ship. Same night. Completely different odds. Six lines of numbers tell that story.

### `.agg()` — several summaries at once

In [120]:
# Give agg a list of what you want
titanic.groupby("class")["fare"].agg(["count", "mean", "min", "max"])

,count,mean,min,max
class,,,,
First,216,84.154687,0.0,512.3292
Second,184,20.662183,0.0,73.5000
Third,491,13.675550,0.0,69.5500


In [121]:
# Different summaries for different columns
titanic.groupby("sex").agg({
    "survived": "mean",
    "age": "mean",
    "fare": "max"
})

,survived,age,fare
sex,,,
female,0.742038,27.915709,512.3292
male,0.188908,30.726645,512.3292


## 7. Missing values — just a peek

We saw on Day 1 that `age` has missing values. Let's confirm and see how it affects us.

In [122]:
print("Missing values per column:")
print(titanic.isna().sum())

Missing values per column:
survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


In [123]:
# Important: Pandas SKIPS missing values in calculations by default
print("Total rows        :", titanic.shape[0])
print("Non-missing ages  :", titanic["age"].count())
print("Average age       :", titanic["age"].mean())
print()
print("The mean was calculated from 714 ages, not 891.")

Total rows        : 891
Non-missing ages  : 714
Average age       : 29.69911764705882

The mean was calculated from 714 ages, not 891.


In [124]:
# You can filter to only rows where age is present
has_age = titanic[titanic["age"].notna()]
print("Rows with a known age:", has_age.shape[0])

# Or only rows where it's missing
no_age = titanic[titanic["age"].isna()]
print("Rows with unknown age:", no_age.shape[0])

Rows with a known age: 714
Rows with unknown age: 177


We are only **counting and noticing** missing data today.

**Deciding what to do about it** — drop the rows? fill in a guess? — is the whole
topic of Week 6. Park it for now, but remember: real data is always missing something.

## Wrap-up — today's toolkit

```python
# Rows
df.loc[0:3]              # by label   (INCLUSIVE stop)
df.iloc[0:3]             # by position (EXCLUSIVE stop)
df.loc[0:3, ["a","b"]]   # rows and columns

# Filtering — the big one
df[df["age"] > 30]
df[(df["sex"] == "female") & (df["survived"] == 1)]     # AND
df[(df["class"] == "First") | (df["class"] == "Second")] # OR
df[~(df["survived"] == 1)]                               # NOT
df[df["class"].isin(["First", "Second"])]
df[df["age"].between(13, 19)]

# Sorting
df.sort_values(by="fare", ascending=False)
df.sort_values(by=["class", "fare"], ascending=[True, False])
df.nlargest(5, "fare")    df.nsmallest(5, "age")

# Columns
df["new"] = df["a"] + df["b"]
df.rename(columns={"old": "new"})
df.drop(columns=["a", "b"])
df.copy()

# Grouping
df.groupby("sex")["survived"].mean()
df.groupby(["sex", "class"])["survived"].mean()
df.groupby("class")["fare"].agg(["count", "mean", "max"])

# Missing (counting only — fixing is Week 6)
df.isna().sum()    df["age"].notna()    df["age"].isna()
```

### Remember these three
1. `.loc` includes the stop, `.iloc` does not
2. Use `&` `|` `~`, and bracket **every** condition
3. `.copy()` before you modify

### Next class (Day 3)
**Build & Ship** — the Titanic Data Explorer project, pushed to your GitHub.

### Practice
Complete **`week5_day2_practice.ipynb`** before Day 3. Do not skip it — Day 3
assumes you can filter and group without looking it up.

See you next class. 🚀